# 06 - Hyper-parameter Tuning (GridSearchCV)

Project: **RoadEye - Traffic Sign Recognition Using HOG Features and SVM** (Machine Learning Innovative Assignment).

> Run cells top to bottom. All numbers are computed when the cell runs; nothing is typed in by hand.

In [ ]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
from src import config, pipeline as P
np.random.seed(config.SEED)                      # reproducibility
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("RUN MODE:", config.RUN_MODE, "| seed:", config.SEED)

In [ ]:
def show(name, width=900):
    """Display a saved figure from results/figures/."""
    display(Image(filename=str(config.FIGURES_DIR / name), width=width))

def metrics(name):
    return json.loads((config.METRICS_DIR / name).read_text())

## Leakage-safe grid search
* Only the **training split** is used.
* `StandardScaler` is inside the CV `Pipeline` (refit in each fold).
* Folds are `StratifiedGroupKFold` over sign tracks.
* Scoring: macro-F1.
* To keep it affordable the grid search runs on a stratified, group-aware subsample of the training split (`TUNE_MAX_SAMPLES`); the best configuration is then refit on the **full** training split and scored on validation.

In [ ]:
print('SVM grid :', config.SVM_GRID)
print('KNN grid :', config.KNN_GRID)
print('Tree grid:', config.TREE_GRID)
print('CV folds:', config.CV_FOLDS, '| scoring:', config.TUNING_SCORING, '| subsample:', config.TUNE_MAX_SAMPLES)

In [ ]:
data = P.stage_prepare(save_figures=False)
for f, fn in [('baseline_svm.json', P.stage_baseline), ('svm_default.json', P.stage_compare)]:
    if not (config.METRICS_DIR / f).exists():
        fn(data)
results = P.stage_tune(data, tune_others=True, n_jobs=2)

In [ ]:
rows = []
for k, r in results.items():
    rows.append({'model': k, 'best_params': r['best_params'], 'best_cv_f1': round(r['best_cv_score'], 4),
                 'val_f1_before': round(r['validation_f1_macro_before_tuning'], 4), 'val_f1_after': round(r['validation_f1_macro_after_tuning'], 4),
                 'n_fits': r['n_fits'], 'search_time_s': round(r['search_time_seconds'], 1), 'subsampled': r['subsampled']})
display(pd.DataFrame(rows))

In [ ]:
display(pd.DataFrame(results['svm']['cv_results']).sort_values('rank_test_score').head(10))
show('grid_search_svm.png', 500)

## Saved artefacts
`models/svm/` now holds `best_svm_model.joblib`, `scaler.joblib`, `best_parameters.json`, `label_mapping.json`, `model_metadata.json`.

In [ ]:
print(sorted(p.name for p in (config.MODELS_DIR / 'svm').iterdir()))
display(pd.Series(json.loads((config.MODELS_DIR / 'svm' / 'best_parameters.json').read_text())))

Final configuration = the SVM selected by GridSearchCV (declared in advance as the project's proposed model). If the tuned KNN or Tree has a higher validation macro-F1, that is reported honestly in the comparison table.